In [1]:
print('hello world')

hello world


In [1]:
import json
import faiss
import numpy as np
import requests
from sentence_transformers import SentenceTransformer

c:\Users\acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3677.24it/s]


In [3]:
# Load schema documents
with open("schema_docs.json", "r") as f:
    docs = json.load(f)

texts = [d["text"] for d in docs]

In [4]:
# Create embeddings
embeddings = model.encode(texts)

In [5]:
# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [6]:
# ---- RAG Retrieval ----
def retrieve_context(query, k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    
    results = [texts[i] for i in indices[0]]
    return "\n".join(results)


In [9]:
import ollama

In [10]:
# ---- LLM Call (Ollama) ----
def call_llm(prompt):
    response=ollama.chat(
        model="llama3",
        messages=[{"role":"user","content":prompt}]
    )
    return response['message']['content']

In [11]:
# ---- Prompt Template ----
def build_prompt(query, context):
    return f"""
You are an expert SQL generator.

Use the following database schema and rules:
{context}

Rules:
- Only generate SELECT queries
- Use correct joins
- Do not hallucinate tables or columns

User Question:
{query}

Output format:
SQL Query:
<query>

Explanation:
<explanation>
"""

In [12]:
# ---- Main Pipeline ----
def generate_sql(query):
    context = retrieve_context(query)
    prompt = build_prompt(query, context)
    
    response = call_llm(prompt)
    return response

In [13]:
# ---- Run Example ----
if __name__ == "__main__":
    user_query = input("Enter your query: ")
    
    result = generate_sql(user_query)
    
    print("\n=== RESULT ===")
    print(result)


=== RESULT ===
Here is the SQL query to answer the user question:

**SQL Query:**
```
SELECT u.id, u.name, SUM(oi.quantity * oi.price) AS total_revenue
FROM orders o
JOIN order_items oi ON o.id = oi.order_id
JOIN users u ON o.user_id = u.id
WHERE MONTH(o.order_date) = MONTH(CURRENT_DATE - INTERVAL 1 MONTH)
GROUP BY u.id, u.name
ORDER BY total_revenue DESC
LIMIT 3;
```

**Explanation:**
To find the top 3 customers by revenue last month, we need to join the `orders` table with the `order_items` table to calculate the revenue for each order. Then, we join the result with the `users` table to get the customer information.

We filter the results to only include orders from last month using the `MONTH` function and a date arithmetic operation. The rest of the query is straightforward: group by the user ID and name, sort by total revenue in descending order, and limit the results to the top 3 customers.
